# LaunchGrid human b-roll — Wan 2.1 (1.3B) on a free GPU

Runs on **Google Colab free tier (T4 16GB)** or **Kaggle (30 GPU-hrs/week)**.

1. Runtime → Change runtime type → **T4 GPU**
2. Upload `prompts.json` (left sidebar → Files) — or edit the inline fallback below
3. Run all cells. Each 5s portrait clip takes **~10–20 min on a T4** — queue them and walk away
4. Last cell zips `out/` → download → drop the MP4s into `launchgrid-ads/public/clips/`

Faster-but-rougher alternative: swap the model cell for **LTX-Video 2B distilled** (~1–2 min/clip on T4) — see the comment in the model cell.

In [ ]:
!nvidia-smi
%pip install -q "diffusers>=0.33" transformers accelerate ftfy imageio imageio-ffmpeg sentencepiece

In [ ]:
import torch
from diffusers import AutoencoderKLWan, WanPipeline

# T4 has no bfloat16 — use float16. cpu_offload keeps peak VRAM ~8GB.
MODEL = "Wan-AI/Wan2.1-T2V-1.3B-Diffusers"
vae = AutoencoderKLWan.from_pretrained(MODEL, subfolder="vae", torch_dtype=torch.float32)
pipe = WanPipeline.from_pretrained(MODEL, vae=vae, torch_dtype=torch.float16)
pipe.enable_model_cpu_offload()

# --- ALTERNATIVE (much faster, slightly less realistic humans): ---
# from diffusers import LTXPipeline
# pipe = LTXPipeline.from_pretrained("Lightricks/LTX-Video", torch_dtype=torch.float16)
# pipe.enable_model_cpu_offload()

In [ ]:
import json, os
from pathlib import Path
from diffusers.utils import export_to_video

os.makedirs("out", exist_ok=True)

if Path("prompts.json").exists():
    spec = json.loads(Path("prompts.json").read_text())
else:  # inline fallback — edit freely
    spec = {
        "defaults": {"width": 480, "height": 832, "num_frames": 81, "fps": 16, "steps": 30, "guidance": 5.0},
        "negative": "text, watermark, distorted hands, deformed face, low quality, cartoon",
        "clips": [{"id": "test", "prompt": "Cinematic close-up, dark bedroom at night, a smartphone lights up with warm glow, photorealistic"}],
    }

d = spec["defaults"]
for clip in spec["clips"]:
    dst = f"out/{clip['id']}.mp4"
    if Path(dst).exists():
        print("skip", dst); continue
    print("generating", clip["id"], "...")
    frames = pipe(
        prompt=clip["prompt"],
        negative_prompt=spec["negative"],
        width=clip.get("width", d["width"]),
        height=clip.get("height", d["height"]),
        num_frames=clip.get("num_frames", d["num_frames"]),
        num_inference_steps=clip.get("steps", d["steps"]),
        guidance_scale=clip.get("guidance", d["guidance"]),
        generator=torch.Generator("cuda").manual_seed(clip.get("seed", 42)),
    ).frames[0]
    export_to_video(frames, dst, fps=clip.get("fps", d["fps"]))
    print("wrote", dst)

In [ ]:
# Zip + download
!cd out && zip -r ../launchgrid-clips.zip .
try:
    from google.colab import files
    files.download("launchgrid-clips.zip")
except ImportError:
    print("On Kaggle: download launchgrid-clips.zip from the output panel")